# Tierkreis and HPC

Tierkreis supports execution in HPC environments through executors natively.
Before we take at a look at the executor lets define a problem to run.

For this example we are using symbolic circuits, which are supported through the `tkr_pytket_worker`.
As before were defining inputs and outputs:

In [ ]:
from typing import NamedTuple

from tierkreis.data.models import TKR, OpaqueType


class SymbolicCircuitsInputs(NamedTuple):
    a: TKR[float]
    b: TKR[float]
    c: TKR[float]
    ansatz: TKR[OpaqueType["pytket._tket.circuit.Circuit"]]  # noqa: F821


class SymbolicCircuitsOutputs(NamedTuple):
    expectation: TKR[float]

and defining a graph under the assumption that our circuit will have three free symbols

In [ ]:
from aer_worker import submit_single
from pytket_worker import (
    add_measure_all,
    expectation,
    optimise_phase_gadgets,
)
from substitution_worker import substitute
from tierkreis.builder import Graph, Workflow


def symbolic_execution() -> Workflow:
    """A graph that substitutes 3 parameters into a circuit and gets an expectation value."""
    g = Graph(SymbolicCircuitsInputs, SymbolicCircuitsOutputs)
    a = g.inputs.a
    b = g.inputs.b
    c = g.inputs.c
    ansatz = g.inputs.ansatz
    n_shots = g.const(100)

    substituted_circuit = g.task(substitute(a=a, b=b, c=c, circuit=ansatz))
    measurement_circuit = g.task(add_measure_all(circuit=substituted_circuit))

    compiled_circuit = g.task(optimise_phase_gadgets(circuit=measurement_circuit))
    backend_result = g.task(submit_single(circuit=compiled_circuit, n_shots=n_shots))
    av = g.task(expectation(backend_result=backend_result))

    return g.finish_with_outputs(SymbolicCircuitsOutputs(expectation=av))

defining the ansatz:

In [ ]:
from pytket._tket.circuit import Circuit, fresh_symbol


def build_ansatz() -> Circuit:
    a = fresh_symbol("a")
    b = fresh_symbol("b")
    c = fresh_symbol("c")
    circ = Circuit(4)
    circ.CX(0, 1)
    circ.CX(1, 2)
    circ.CX(2, 3)
    circ.Rz(a, 3)
    circ.CX(2, 3)
    circ.CX(1, 2)
    circ.CX(0, 1)
    circ.Rz(b, 0)
    circ.CX(0, 1)
    circ.CX(1, 2)
    circ.CX(2, 3)
    circ.Rz(c, 3)
    circ.CX(2, 3)
    circ.CX(1, 2)
    circ.CX(0, 1)
    return circ

Most HPC systems use a shared file system which can be used as a storage.
Make sure the the checkpoints live on there!

In [ ]:
from pathlib import Path

from tierkreis.runtime import load_runtime_config, new_from_config

# Configure shared File assets, persistent SQLite state, and an Hpc
# executor using scheduler = "pjsub" in this file.
config = load_runtime_config(Path("runtime-hpc.toml"))

The runtime configuration routes ordinary installed workers to a subprocess executor and HPC tasks to the configured PJSUB executor. Scheduler resources such as nodes, memory, walltime, account, and queue belong in the selected resource profile rather than in Python executor objects.

In [ ]:
runtime = await new_from_config(config)

We can now run the example after defining the inputs.
Make sure that the code is running on an HPC system and you have selected the correct group and executor!

In [ ]:
workflow = symbolic_execution()
with runtime:
    workflow_id = await runtime.save_workflow("symbolic_circuits", workflow)
    run_id = await runtime.start_new_run(
        workflow_id,
        {
            "ansatz": build_ansatz().to_dict(),
            "a": 0.2,
            "b": 0.55,
            "c": 0.75,
        },
        profile="fugaku",
    )
    await runtime.wait_for(run_id)
    output = await runtime.get_outputs(run_id)